In [3]:
from datetime import date
import pandas as pd
import matplotlib.pyplot as plt

from pairs_trading.data.universe import get_sp500_constituents, load_constituents_from_csv
from pairs_trading.data.stooq_loader import fetch_stooq_ohlcv, save_raw_prices
from pairs_trading.data.cleaning import clean_ohlcv, align_prices
from pairs_trading.features.returns import compute_daily_returns
from pairs_trading.selection.pair_selector import PairSelector
from pairs_trading.strategies.bollinger import BollingerStrategy
from pairs_trading.strategies.ou import OUForecastStrategy
from pairs_trading.strategies.copula import CopulaStrategy
from pairs_trading.strategies.cointegration import CointegrationStrategy
from pairs_trading.backtest.engine import BacktestEngine
from pairs_trading.backtest.metrics import (
    cumulative_return,
    annualized_return,
    volatility,
    max_drawdown,
    sharpe_ratio,
    trade_count,
)

# =========================
# Paths / settings
# =========================

from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists() or (parent / "README.md").exists():
            return parent
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
UNIVERSE_DIR = DATA_DIR / "universe"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"

import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))

for d in [RAW_DIR, UNIVERSE_DIR, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

AS_OF = "2023-03-31"
START = date(2022, 3, 1)
END = date(2023, 3, 31)
INTERVAL = "1d"
MAX_SYMBOLS = 100          # keep small for the first run
PCA_VARIANCE = 0.8
MIN_SAMPLES = 3

# =========================
# Helpers
# =========================
def save_fig(name: str):
    path = FIGURES_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved figure -> {path}")

def load_or_download_universe(as_of: str) -> pd.DataFrame:
    cached = UNIVERSE_DIR / "sp500_constituents.csv"
    if cached.exists():
        print("Loading cached universe CSV")
        return load_constituents_from_csv(str(cached))

    print("Downloading universe...")
    df = get_sp500_constituents(as_of)
    df.to_csv(cached, index=False)
    print(f"saved universe -> {cached}")
    return df

def load_or_download_price(symbol: str) -> pd.DataFrame:
    safe = symbol.replace("/", "-").replace(".", "_")
    path = RAW_DIR / f"{safe}.{INTERVAL}.csv"

    if path.exists() and path.stat().st_size > 0:
        df = pd.read_csv(path, parse_dates=["date"], index_col="date")
        return clean_ohlcv(df)

    try:
        df = fetch_stooq_ohlcv(symbol, INTERVAL, START, END)
        if df is None or df.empty:
            return pd.DataFrame()
        save_raw_prices(df, str(path))
        return clean_ohlcv(df)
    except Exception as e:
        print(f"[WARN] {symbol}: {e}")
        return pd.DataFrame()

def build_or_load_processed_prices(universe: pd.DataFrame) -> pd.DataFrame:
    path = PROCESSED_DIR / "prices.parquet"
    if path.exists():
        print("Loading cached processed dataset")
        return pd.read_parquet(path)

    symbols = universe["Symbol"].dropna().astype(str).tolist()[:MAX_SYMBOLS]
    print(f"Downloading up to {len(symbols)} symbols")

    data = {}
    for sym in symbols:
        df = load_or_download_price(sym)
        if df is not None and not df.empty:
            data[sym] = df

    aligned = align_prices(data)
    aligned.to_parquet(path)
    print(f"saved processed prices -> {path}")
    return aligned

def summarize_backtest(result: pd.DataFrame, initial_capital: float = 100000.0) -> dict:
    equity = result["equity_curve"]
    trades = result.attrs.get("trades", pd.DataFrame())

    return {
        "final_equity": float(equity.iloc[-1]),
        "cumulative_return": cumulative_return(equity),
        "annualized_return": annualized_return(equity),
        "volatility": volatility(equity),
        "max_drawdown": max_drawdown(equity),
        "sharpe_ratio": sharpe_ratio(equity),
        "trade_count": trade_count(trades),
    }

# =========================
# 1) Universe
# =========================
universe = load_or_download_universe(AS_OF)
display(universe.head())

# =========================
# 2) Prices
# =========================
prices = build_or_load_processed_prices(universe)
display(prices.head())

# =========================
# 3) Pair selection
# =========================
returns = compute_daily_returns(prices)

selector = PairSelector(
    target_variance=PCA_VARIANCE,
    min_samples=MIN_SAMPLES,
)
selector.fit(prices)
selected_pairs = selector.select_pairs()

print("clusters:", len(selector.clusters))
print("candidates:", len(selector.generate_candidates()))

pairs_path = TABLES_DIR / "selected_pairs.csv"
selected_pairs.to_csv(pairs_path, index=False)
print(f"saved selected pairs -> {pairs_path}")

display(selected_pairs.head(10))

# =========================
# 4) Pick one pair to test strategies
# =========================
if selected_pairs.empty:
    raise RuntimeError("No pairs selected. Try increasing MAX_SYMBOLS or relaxing filters.")

pair_row = selected_pairs.iloc[0]
left = pair_row["asset_left"]
right = pair_row["asset_right"]

pair_data = prices[[left, right]].dropna()
print(f"Testing pair: {left} / {right}")
display(pair_data.head())

# =========================
# 5) Run strategies
# =========================
engine = BacktestEngine(
    initial_capital=100000.0,
    notional_per_trade=1.0,
    transaction_cost_bps=1.0,
    slippage_bps=1.0,
)

strategies = {
    "bollinger": BollingerStrategy(window=20, num_std=2.0, exit_std=0.5),
    "ou": OUForecastStrategy(window=20, confidence=0.98, n_paths=20000, entry_z=1.0, exit_z=0.2),
    "copula": CopulaStrategy(entry_quantile=0.15, exit_quantile=0.5),
    "cointegration": CointegrationStrategy(window=20, entry_z=2.0, exit_z=0.5),
}

results = {}
metrics_rows = []

for name, strat in strategies.items():
    print(f"Running strategy: {name}")
    strat.fit(pair_data)
    res = engine.run(strat, pair_data)
    results[name] = res

    metrics = summarize_backtest(res)
    metrics["strategy"] = name
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).set_index("strategy").sort_values(
    by="cumulative_return", ascending=False
)

metrics_path = TABLES_DIR / "strategy_metrics.csv"
metrics_df.to_csv(metrics_path)
print(f"saved metrics -> {metrics_path}")
display(metrics_df)

# =========================
# 6) Chart 1: equity curve comparison
# =========================
plt.figure(figsize=(12, 6))
for name, res in results.items():
    equity = res["equity_curve"] / res["equity_curve"].iloc[0]
    plt.plot(equity.index, equity.values, label=name)
plt.title(f"Equity Curve Comparison: {left} / {right}")
plt.xlabel("Date")
plt.ylabel("Normalized Equity")
plt.legend()
save_fig("equity_curve_comparison.png")
plt.show()

# =========================
# 7) Chart 2: drawdown comparison
# =========================
plt.figure(figsize=(12, 6))
for name, res in results.items():
    equity = res["equity_curve"]
    dd = equity / equity.cummax() - 1.0
    plt.plot(dd.index, dd.values, label=name)
plt.title(f"Drawdown Comparison: {left} / {right}")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.legend()
save_fig("drawdown_comparison.png")
plt.show()

# =========================
# 8) Chart 3: spread + signals for best strategy
# =========================
best_strategy = metrics_df.index[0]
best_res = results[best_strategy]

plt.figure(figsize=(12, 6))
plt.plot(best_res.index, best_res["spread"], label="spread")
plt.plot(best_res.index, best_res["signal"], label="signal")
plt.title(f"Spread and Signal: {best_strategy} on {left} / {right}")
plt.xlabel("Date")
plt.legend()
save_fig("best_strategy_spread_signal.png")
plt.show()

# =========================
# 9) Chart 4: metric comparison bar chart
# =========================
plt.figure(figsize=(12, 6))
metrics_df["cumulative_return"].plot(kind="bar")
plt.title("Cumulative Return by Strategy")
plt.xlabel("Strategy")
plt.ylabel("Cumulative Return")
save_fig("cumulative_return_bar.png")
plt.show()

# =========================
# 10) Optional: save each strategy result
# =========================
for name, res in results.items():
    out = TABLES_DIR / f"{name}_backtest.csv"
    res.to_csv(out)
    print(f"saved backtest -> {out}")

PROJECT_ROOT = D:\sem 8\BTP2 - Pairs Trading
Loading cached universe CSV


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,hideFounded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


Loading cached processed dataset


,MMM,AOS,ABT,ABBV,ACN,ADBE,AMD,AES,AFL,A,...,ALGN,ALLE,LNT,ALL,GOOGL,GOOG,MO,AMZN,AMCR,AEE
date,,,,,,,,,,,,,,,,,,,,,
2022-03-01,140.204,65.6445,116.629,143.631,307.961,466.68,113.83,20.2479,57.3546,131.260,...,500.97,112.835,55.9777,117.530,133.578,134.168,48.6426,151.142,46.7488,82.6037
2022-03-02,142.596,68.3740,118.790,145.456,314.855,471.18,118.28,20.9733,58.9826,132.486,...,496.14,115.532,56.3390,121.472,134.086,134.752,49.8122,152.052,47.4128,83.7737
2022-03-03,143.643,67.6644,119.394,146.270,315.470,459.08,111.98,21.0217,59.9830,136.472,...,477.45,115.868,57.7368,123.238,133.416,134.308,50.1519,147.898,47.1222,85.0496
2022-03-04,142.122,67.3794,119.830,146.417,310.738,452.13,108.41,21.3750,58.9433,133.213,...,464.48,115.452,59.2232,125.071,131.429,132.122,50.4531,145.641,46.8732,87.2119
2022-03-07,138.781,66.3244,117.312,145.338,301.690,437.97,102.95,21.4234,57.8939,129.686,...,435.57,112.933,59.3886,122.737,125.922,126.464,49.8496,137.453,44.7972,87.2610


clusters: 1
candidates: 91
saved selected pairs -> D:\sem 8\BTP2 - Pairs Trading\outputs\tables\selected_pairs.csv


""


RuntimeError: No pairs selected. Try increasing MAX_SYMBOLS or relaxing filters.